# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GazalaNK/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The ranked queue from w05/w06 (Random Forest predictions on the client-grouped test split) is turned into an action playbook. Each page gets: a rank, a reason code, and a suggested action.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
playbook = test.copy()
playbook["reason_code"] = "ctr_below_tier_expectation"
playbook["action"] = "review_metadata_and_snippet"
playbook = playbook.sort_values("model_score", ascending=False).reset_index(drop=True)
playbook["rank"] = playbook.index + 1

playbook[["rank","content_hash_id","impressions","clicks","ctr","avg_position","model_score","reason_code","action"]].head(20)

NameError: name 'test' is not defined

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use: this playbook ranks pages for a human reviewer's limited time — it prioritizes which pages to look at first for a possible metadata/snippet review, given that they're visible (good position, sufficient impressions) but under-capturing clicks relative to peers at the same position.

Limits: this is observed on one month of data (March 2026) for one lane; it does not establish that a rewrite will improve CTR, and it does not account for seasonal demand shifts, recent content changes not yet reflected in ranking, or SERP feature changes (e.g. a new "People Also Ask" box reducing clicks regardless of the page's own quality).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick check: confirm the minimum impression threshold actually being enforced
print("Minimum impressions threshold used:", MIN_IMPRESSIONS if "MIN_IMPRESSIONS" in dir() else 100)
print("Rows excluded below threshold:", (page["impressions"] < 100).sum())

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human review required for every recommendation — this is a candidate list, not an automated action queue.

No-go cases (should NOT be automated):

Do not auto-rewrite metadata/titles based on this score alone
Do not treat a high rank as proof the current metadata is "bad" — it may reflect seasonal timing, a recent unindexed change, or normal volatility
Do not apply this scoring to pages below the minimum impression threshold (too noisy to trust)
Do not use this to make client-facing claims about causing a CTR improvement without a genuine before/after experiment

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Confirm no page below the noise threshold made it into the top 20
below_threshold_in_top20 = (playbook.head(20)["impressions"] < 100).sum()
print("Top-20 rows below minimum impression threshold (should be 0):", below_threshold_in_top20)


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Monitoring: re-run the scoring pipeline monthly on a new mid-panel month, and track whether the same pages keep appearing at the top (a sign of a persistent issue) vs. rotating (a sign of noise or seasonal effects).

Retrain triggers: retrain the model if — position-tier CTR medians shift meaningfully month over month (suggesting SERP-wide behavior change), Precision@20 on a fresh month drops notably below the validated baseline, or a new content type/client segment appears that wasn't represented in the training data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show current position-tier CTR medians — the number to compare against in future months
print(page.groupby("position_tier")["ctr"].median())


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, json

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

playbook.to_csv("work/outputs/action_playbook.csv", index=False)

metrics = {
    "baseline_precision_at_20": precision_at_k(test, "baseline_score"),
    "model_precision_at_20": precision_at_k(test, "model_score"),
    "test_rows": test.shape[0],
    "month": "2026-03"
}
with open("work/outputs/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Exported playbook and metrics.")
print(metrics)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.